In [1]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_VERSION = "forecast_lgbm_v1"
FEATURE_COLUMNS = ["day_of_week", "is_weekend", "month", "day_of_month", "is_festive_season",
                   "store_type_code", "sales_lag_7", "sales_lag_14", "sales_lag_28",
                   "sales_mean_7d", "sales_mean_28d"]

conn = sqlite3.connect(DB_PATH)
model = lgb.Booster(model_file=str(MODELS_DIR / f"{MODEL_VERSION}.txt"))

test = pd.read_sql("SELECT * FROM features_daily_sales WHERE date >= '2025-09-01'",
                   conn, parse_dates=["date"]).reset_index(drop=True)
test["prediction"] = model.predict(test[FEATURE_COLUMNS])

saved = pd.read_sql("SELECT store_id, date, pred_lightgbm FROM forecast_predictions", conn, parse_dates=["date"])
check = test.merge(saved, on=["store_id", "date"])
print("Test rows:", len(test))
print("Largest difference from Step 6 predictions: Rs", round((check["prediction"] - check["pred_lightgbm"]).abs().max(), 3))

Test rows: 1464
Largest difference from Step 6 predictions: Rs 0.005


In [2]:
contributions = model.predict(test[FEATURE_COLUMNS], pred_contrib=True)

shap_values = pd.DataFrame(contributions[:, :-1], columns=FEATURE_COLUMNS)
base_value = contributions[0, -1]

print(f"Starting value (average prediction): Rs {base_value:,.0f}")
print("Pieces add up to every prediction:", np.allclose(contributions.sum(axis=1), test["prediction"]))

mean_abs = shap_values.abs().mean().sort_values(ascending=False)
shap_global = pd.DataFrame({"feature": mean_abs.index, "mean_abs_shap_rs": mean_abs.values.round(0)})
shap_global["share_pct"] = (100 * shap_global["mean_abs_shap_rs"] / shap_global["mean_abs_shap_rs"].sum()).round(1)

shap_global

Starting value (average prediction): Rs 90,697
Pieces add up to every prediction: True


,feature,mean_abs_shap_rs,share_pct
0,sales_mean_28d,20222.0,30.5
1,day_of_week,11944.0,18.0
2,store_type_code,9339.0,14.1
3,is_festive_season,5273.0,8.0
4,sales_mean_7d,4184.0,6.3
5,month,4044.0,6.1
6,is_weekend,3095.0,4.7
7,sales_lag_7,2990.0,4.5
8,sales_lag_28,2138.0,3.2
9,sales_lag_14,1774.0,2.7


In [3]:
test["error"] = test["sales"] - test["prediction"]

examples = {
    "Flagship store, festive Saturday": test[(test["store_type"] == "Flagship") & (test["date"] == "2025-10-25")].index[0],
    "Express store, ordinary Monday": test[(test["store_type"] == "Express") & (test["date"] == "2025-09-15")].index[0],
    "Largest miss in test period": test["error"].abs().idxmax(),
}

example_rows = []
for label, i in examples.items():
    row = test.loc[i]
    print(f"\n{label}: store {row['store_id']}, {row['date'].date()}")
    print(f"  Actual Rs {row['sales']:,.0f} | Predicted Rs {row['prediction']:,.0f}")
    print(f"  Starting value Rs {base_value:,.0f}, then:")

    pieces = shap_values.loc[i].sort_values(key=abs, ascending=False)
    for feature, value in pieces.head(5).items():
        print(f"    {feature:<18} {value:+12,.0f}   (feature value: {row[feature]:,.0f})")
        example_rows.append({"example": label, "store_id": row["store_id"], "date": row["date"].strftime("%Y-%m-%d"),
                             "actual_sales": round(row["sales"], 2), "predicted_sales": round(row["prediction"], 2),
                             "feature": feature, "feature_value": round(row[feature], 2),
                             "shap_contribution_rs": round(value, 2)})

shap_examples = pd.DataFrame(example_rows)


Flagship store, festive Saturday: store S005, 2025-10-25
  Actual Rs 242,730 | Predicted Rs 247,665
  Starting value Rs 90,697, then:
    sales_mean_28d          +62,718   (feature value: 184,140)
    day_of_week             +28,931   (feature value: 5)
    store_type_code         +22,642   (feature value: 2)
    sales_mean_7d           +16,721   (feature value: 205,699)
    is_festive_season        +9,075   (feature value: 1)

Express store, ordinary Monday: store S001, 2025-09-15
  Actual Rs 24,176 | Predicted Rs 54,725
  Starting value Rs 90,697, then:
    sales_mean_28d          -12,531   (feature value: 59,823)
    day_of_week             -10,267   (feature value: 0)
    store_type_code          -6,566   (feature value: 0)
    is_weekend               -1,685   (feature value: 0)
    sales_mean_7d            -1,445   (feature value: 53,952)

Largest miss in test period: store S002, 2025-10-14
  Actual Rs 1,600,240 | Predicted Rs 119,547
  Starting value Rs 90,697, then:
    sales_

In [4]:
rows = []
for segment_col in ["store_type", "region"]:
    for segment_value, group in test.groupby(segment_col):
        rows.append({
            "segment_type": segment_col,
            "segment": segment_value,
            "store_days": len(group),
            "avg_actual_rs": round(group["sales"].mean()),
            "avg_predicted_rs": round(group["prediction"].mean()),
            "bias_pct": round(100 * (group["prediction"].sum() - group["sales"].sum()) / group["sales"].sum(), 1),
            "WAPE_pct": round(100 * group["error"].abs().sum() / group["sales"].sum(), 1),
        })

segment_check = pd.DataFrame(rows)

overall_bias = 100 * (test["prediction"].sum() - test["sales"].sum()) / test["sales"].sum()
overall_wape = 100 * test["error"].abs().sum() / test["sales"].sum()
print(f"Overall: bias {overall_bias:+.1f}%, WAPE {overall_wape:.1f}%")

BIAS_LIMIT = 5.0
WAPE_GAP_LIMIT = 10.0
segment_check["bias_flag"] = segment_check["bias_pct"].abs() > BIAS_LIMIT
segment_check["wape_gap_flag"] = (segment_check["WAPE_pct"] - overall_wape) > WAPE_GAP_LIMIT / 2

segment_check

Overall: bias +1.2%, WAPE 33.6%


,segment_type,segment,store_days,avg_actual_rs,avg_predicted_rs,bias_pct,WAPE_pct,bias_flag,wape_gap_flag
0,store_type,Express,732,83362,82780,-0.7,38.4,False,False
1,store_type,Flagship,244,182132,184245,1.2,27.2,False,False
2,store_type,Standard,488,118179,122041,3.3,33.5,False,False
3,region,East,366,99814,105720,5.9,34.0,True,False
4,region,North,366,121349,119527,-1.5,34.2,False,False
5,region,South,366,93650,94267,0.7,38.7,False,True
6,region,West,366,130903,131597,0.5,29.1,False,False


In [5]:
created_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

notes = pd.DataFrame([
    ("Bias", "East region is over-forecast by about 6% in the test period (limit 5%). "
             "Monitor after retraining; if it persists, East stores would receive slightly too much stock."),
    ("Accuracy gap", "Express stores have higher error (38%) than Flagship stores (27%). "
                     "This is day-to-day noise in small stores, not a systematic lean (bias -0.7%)."),
    ("Unpredictable events", "Single huge orders (for example Rs 16 lakh at S002 on 2025-10-14) cannot be forecast. "
                             "They are caught by the anomaly workflow instead (Step 7)."),
    ("Scope of fairness", "The model uses no personal or customer data, only store and calendar information, "
                          "so fairness is checked across stores, store types and regions."),
    ("Explainability method", "SHAP values are computed with LightGBM's built-in TreeSHAP (pred_contrib=True); "
                              "the contributions add up exactly to each prediction."),
    ("Synthetic data", "All findings come from synthetic data and must be re-checked on real data."),
], columns=["topic", "note"])

tables = {
    "forecast_shap_global": shap_global,
    "forecast_shap_examples": shap_examples,
    "forecast_segment_check": segment_check,
    "forecast_bias_notes": notes,
}
for name, df in tables.items():
    df = df.copy()
    df["model_version"] = MODEL_VERSION
    df["created_at"] = created_at
    df.to_sql(name, conn, if_exists="replace", index=False)
    df.to_csv(REPORTS_DIR / f"{name}.csv", index=False)
    print(f"Saved {name}: {len(df)} rows")

pd.set_option("display.max_colwidth", None)
notes

Saved forecast_shap_global: 11 rows
Saved forecast_shap_examples: 15 rows
Saved forecast_segment_check: 7 rows
Saved forecast_bias_notes: 6 rows


,topic,note
0,Bias,"East region is over-forecast by about 6% in the test period (limit 5%). Monitor after retraining; if it persists, East stores would receive slightly too much stock."
1,Accuracy gap,"Express stores have higher error (38%) than Flagship stores (27%). This is day-to-day noise in small stores, not a systematic lean (bias -0.7%)."
2,Unpredictable events,Single huge orders (for example Rs 16 lakh at S002 on 2025-10-14) cannot be forecast. They are caught by the anomaly workflow instead (Step 7).
3,Scope of fairness,"The model uses no personal or customer data, only store and calendar information, so fairness is checked across stores, store types and regions."
4,Explainability method,SHAP values are computed with LightGBM's built-in TreeSHAP (pred_contrib=True); the contributions add up exactly to each prediction.
5,Synthetic data,All findings come from synthetic data and must be re-checked on real data.


In [7]:
conn = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT * FROM api_forecast_log", conn)

,store_id,date,predicted_sales,model_version,predicted_at
0,S009,2026-01-03,197656.76,forecast_lgbm_v1,2026-11-02T10:41:54+00:00
1,S009,2026-01-03,197656.76,forecast_lgbm_v1,2026-11-02T10:50:45+00:00
2,S009,2026-01-03,197656.76,forecast_lgbm_v1,2026-11-02T10:51:36+00:00
3,S005,2025-10-25,247664.82,forecast_lgbm_v1,2026-11-02T10:52:58+00:00


In [9]:
conn = sqlite3.connect(DB_PATH)
print(pd.read_sql("SELECT review_status, reviewed_by FROM anomaly_flags WHERE transaction_id = 'T00062458'", conn))
conn.close()

     review_status reviewed_by
0  Confirmed issue      Sameer


In [11]:
conn.close()